In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots

In [ ]:
from generate_data import gen_3d_data

n = 1000
data = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = n, noise = 0.0, seed = 0)[0]
data = data[:, [1, 2, 0]]

camera = dict(eye=dict(x=-1.3, y=-1.4, z=0.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))


fig = plot_scatter_3d(data, color = 'black', camera=camera)
fig.update_layout(scene_camera=camera)
fig.update_layout(width=400, height=450)
fig.show()

In [ ]:
from diffusion_geometry import DiffusionGeometry

quiver_scale = 0.2
arrow_scale = 1.
line_width = 2

dg = DiffusionGeometry.from_point_cloud(data)
a = dg.vector_field_space.zeros()
a.coeffs[2] = 1
fig2 = plot_quiver_3d(data, a.to_ambient(), arrow_scale=arrow_scale, scale=quiver_scale, line_width=line_width)
fig2.update_layout(scene_camera=camera)
fig2.update_layout(width=400, height=450)
fig2.show()

In [ ]:
dg = DiffusionGeometry.from_point_cloud(data)
num_rows = 4
num_cols = 4

specs = [[{"type": "scene"} for _ in range(num_cols)] for _ in range(num_rows)]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.03)

individual_arrow_scale = arrow_scale * np.array([[2, 1, 2],
                                               [2, 1, 2],
                                               [2, 0.5, 2],
                                               [0.4, 1, 0.4]])


for row in range(num_rows):
    f = dg.function_space.zeros()
    f.coeffs[row] = 1
    
    fig1 = plot_scatter_3d(data, color=f.to_ambient(), size=1.5)

    a = dg.vector_field_space.zeros()
    a.coeffs[3*row] = 1
    fig2 = plot_quiver_3d(data, a.to_ambient(), arrow_scale=individual_arrow_scale[row, 0], scale=quiver_scale, line_width=line_width)

    b = dg.vector_field_space.zeros()
    b.coeffs[3*row+1] = 1
    fig3 = plot_quiver_3d(data, b.to_ambient(), arrow_scale=individual_arrow_scale[row, 1], scale=quiver_scale, line_width=line_width)

    c = dg.vector_field_space.zeros()
    c.coeffs[3*row+2] = 1
    fig4 = plot_quiver_3d(data, c.to_ambient(), arrow_scale=individual_arrow_scale[row, 2], scale=quiver_scale, line_width=line_width)

    # Add traces from each figure (this part remains the same)
    fig.add_traces(list(fig1.data), rows=[row+1]*len(fig1.data), cols=[1]*len(fig1.data))
    fig.add_traces(list(fig2.data), rows=[row+1]*len(fig2.data), cols=[2]*len(fig2.data))
    fig.add_traces(list(fig3.data), rows=[row+1]*len(fig3.data), cols=[3]*len(fig3.data))
    fig.add_traces(list(fig4.data), rows=[row+1]*len(fig4.data), cols=[4]*len(fig4.data))

    # Update each subplot's layout to set the camera
    for col in range(1, num_cols + 1):
        fig.update_layout({f'scene{(row*num_cols + col) if (row*num_cols + col) > 1 else ""}': dict(camera=camera)})
    
    # Synchronize axis ranges for bottom row (3D plots)
    all_figs_3d = [fig1, fig2, fig3, fig4]
    x_vals_3d, y_vals_3d, z_vals_3d = [], [], []
    for f in all_figs_3d:
        for t in f.data:
            if hasattr(t, "x") and t.x is not None:
                x_vals_3d.extend([v for v in t.x if v is not None])
            if hasattr(t, "y") and t.y is not None:
                y_vals_3d.extend([v for v in t.y if v is not None])
            if hasattr(t, "z") and t.z is not None:
                z_vals_3d.extend([v for v in t.z if v is not None])

    xrange_3d = [min(x_vals_3d), max(x_vals_3d)]
    yrange_3d = [min(y_vals_3d), max(y_vals_3d)]
    zrange_3d = [min(z_vals_3d), max(z_vals_3d)]

    # Apply to all four columns in this row
    for col in range(1, num_cols + 1):
        scene_index = row * num_cols + col
        scene_key = f"scene{scene_index}" if scene_index > 1 else "scene"
        fig.update_layout({
            scene_key: dict(
                camera=camera,
                xaxis=dict(range=xrange_3d),
                yaxis=dict(range=yrange_3d),
                zaxis=dict(range=zrange_3d),
            )
        })

fig.update_layout(width=800, height=900)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()
fig.write_image("figs/new_output.pdf", scale=1)

In [ ]:
def labels(row, col):
    if col == 1:
        return rf"$\phi_{row}$"
    elif col == 2:
        return rf"$\phi_{row} \nabla x$"
    elif col == 3:
        return rf"$\phi_{row} \nabla y$"
    elif col == 4:
        return rf"$\phi_{row} \nabla z$"
    else:
        return ""

overpic_labels(fig, labels, 
                       stretch_x=1,
                       stretch_y=1,
                       offset_y=12.5,
                       offset_x=1,)

In [ ]:
from generate_data import gen_3d_data

n = 2000
data = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = n, noise = 0.0, seed = 0)[0]
data = data[:, [1, 2, 0]]

camera = dict(eye=dict(x=-1.3, y=-1.4, z=0.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))


fig = plot_scatter_3d(data, color = 'black', camera=camera)
fig.update_layout(scene_camera=camera)
fig.update_layout(width=400, height=450)
fig.show()
dg = DiffusionGeometry.from_point_cloud(data)

In [ ]:
num_rows = 4
num_cols = 4

specs = [[{"type": "scene"} for _ in range(num_cols)] for _ in range(num_rows)]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.03,
)

quiver_scale = 0.1
arrow_scale = 0.3
line_width = 1

n_circle = 30

for row in range(num_rows):
    f = dg.function_space.zeros()
    f.coeffs[row] = 1
    
    fig1 = plot_scatter_3d(data, color=f.to_ambient(), size=1.5)

    a = dg.form_space(2).zeros()
    a.coeffs[3*row] = 1
    fig2 = plot_2form_3d(data, a.to_ambient(), n_circle=n_circle)

    b = dg.form_space(2).zeros()
    b.coeffs[3*row+1] = 1
    fig3 = plot_2form_3d(data, b.to_ambient(), n_circle=n_circle)

    c = dg.form_space(2).zeros()
    c.coeffs[3*row+2] = 1
    fig4 = plot_2form_3d(data, c.to_ambient(), n_circle=n_circle)

    # Add traces from each figure (this part remains the same)
    fig.add_traces(list(fig1.data), rows=[row+1]*len(fig1.data), cols=[1]*len(fig1.data))
    fig.add_traces(list(fig2.data), rows=[row+1]*len(fig2.data), cols=[2]*len(fig2.data))
    fig.add_traces(list(fig3.data), rows=[row+1]*len(fig3.data), cols=[3]*len(fig3.data))
    fig.add_traces(list(fig4.data), rows=[row+1]*len(fig4.data), cols=[4]*len(fig4.data))

    # Update each subplot's layout to set the camera
    for col in range(1, num_cols + 1):
        fig.update_layout({f'scene{(row*num_cols + col) if (row*num_cols + col) > 1 else ""}': dict(camera=camera)})
    
    # Synchronize axis ranges for bottom row (3D plots)
    all_figs_3d = [fig1, fig2, fig3, fig4]
    x_vals_3d, y_vals_3d, z_vals_3d = [], [], []
    for f in all_figs_3d:
        for t in f.data:
            if hasattr(t, "x") and t.x is not None:
                x_vals_3d.extend([v for v in t.x if v is not None])
            if hasattr(t, "y") and t.y is not None:
                y_vals_3d.extend([v for v in t.y if v is not None])
            if hasattr(t, "z") and t.z is not None:
                z_vals_3d.extend([v for v in t.z if v is not None])

    xrange_3d = [min(x_vals_3d), max(x_vals_3d)]
    yrange_3d = [min(y_vals_3d), max(y_vals_3d)]
    zrange_3d = [min(z_vals_3d), max(z_vals_3d)]

    # Apply to all four columns in this row
    for col in range(1, num_cols + 1):
        scene_index = row * num_cols + col
        scene_key = f"scene{scene_index}" if scene_index > 1 else "scene"
        fig.update_layout({
            scene_key: dict(
                camera=camera,
                xaxis=dict(range=xrange_3d),
                yaxis=dict(range=yrange_3d),
                zaxis=dict(range=zrange_3d),
            )
        })

fig.update_layout(width=800, height=900)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()
fig.write_image("figs/new_output.pdf", scale=1)

In [ ]:
def labels(row, col):
    if col == 1:
        return rf"$\phi_{row}$"
    elif col == 2:
        return rf"$\phi_{row} dx \wedge dy$"
    elif col == 3:
        return rf"$\phi_{row} dx \wedge dz$"
    elif col == 4:
        return rf"$\phi_{row} dy \wedge dz$"
    else:
        return ""

overpic_labels(fig, labels, 
                       stretch_x=1,
                       stretch_y=1,
                       offset_y=12.5,
                       offset_x=1,)

3-forms

In [ ]:
from generate_data import gen_3d_data

n = 1000
data = gen_3d_data(kind = 'ball', minor_radius = 1.0, major_radius=2.0, n = n, noise = 0.0, seed = 0)[0]
data = data[:, [1, 2, 0]]

camera = dict(eye=dict(x=-1., y=-1.2, z=0.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))


fig = plot_scatter_3d(data, color = 'black', camera=camera)
fig.update_layout(scene_camera=camera)
fig.update_layout(width=400, height=450)
fig.show()

from diffusion_geometry import DiffusionGeometry
dg = DiffusionGeometry.from_point_cloud(data)

In [ ]:
from plotly.subplots import make_subplots

num_rows = 2
num_cols = 4

specs = [[{"type": "scene"} for _ in range(num_cols)] for _ in range(num_rows)]
fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.0,
)

n_sphere = 12
x_vals_all, y_vals_all, z_vals_all = [], [], []

# ---------- build subplots ----------
for col_idx in range(1, num_cols + 1):
    # function example
    f = dg.function_space.zeros()
    f.coeffs[col_idx - 1] = 1
    fig1 = plot_scatter_3d(data, color=f.to_ambient(), size=2)

    # 3-form example
    a = dg.form_space(3).zeros()
    a.coeffs[col_idx - 1] = 1
    fig2 = plot_3form_3d(data, a.to_ambient(), base_size=0.0, size_scale=30.0, camera=camera)

    # add to subplot grid
    fig.add_traces(list(fig1.data), rows=[1]*len(fig1.data), cols=[col_idx]*len(fig1.data))
    fig.add_traces(list(fig2.data), rows=[2]*len(fig2.data), cols=[col_idx]*len(fig2.data))

    # collect coordinates from both figures
    for subfig in (fig1, fig2):
        for t in subfig.data:
            if hasattr(t, "x") and t.x is not None:
                x_vals_all.extend(t.x)
            if hasattr(t, "y") and t.y is not None:
                y_vals_all.extend(t.y)
            if hasattr(t, "z") and t.z is not None:
                z_vals_all.extend(t.z)

# ---------- compute shared ranges ----------
xrange_3d = [min(x_vals_all), max(x_vals_all)]
yrange_3d = [min(y_vals_all), max(y_vals_all)]
zrange_3d = [min(z_vals_all), max(z_vals_all)]

# ---------- synchronize all 3D axes ----------
camera = dict(
    center=dict(x=0, y=0, z=0),
    eye=dict(x=-1., y=-1.1, z=0.3),
    up=dict(x=0, y=0, z=1),
)

clean_fig(fig)

# apply same ranges and camera to every scene in layout
for scene_key in [k for k in fig.layout if k.startswith("scene")]:
    fig.layout[scene_key].update(
        aspectmode="data",
        camera=camera,
        xaxis=dict(range=xrange_3d),
        yaxis=dict(range=yrange_3d),
        zaxis=dict(range=zrange_3d),
    )

# ---------- layout and export ----------
fig.update_layout(
    width=1000,
    height=500,
    margin=dict(l=0, r=0, t=0, b=0),
)
fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1:
        return rf"$\phi_{col}$"
    elif row == 2:
        return rf"$\phi_{col} dx \wedge dy \wedge dz$"
    else:
        return ""

overpic_labels(fig, labels, 
                       stretch_x=1,
                       stretch_y=0.95,
                       offset_y=12.5,
                       offset_x=-0.5,)